<a href="https://colab.research.google.com/github/zachnelson275/CSE450/blob/main/bikes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [46]:
import numpy as np
import altair as alt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dropout, Flatten, Dense
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import MinMaxScaler

In [47]:
import pandas as pd
bikes = pd.read_csv("https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/bikes.csv")
bikes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112475 entries, 0 to 112474
Data columns (total 12 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   dteday        112475 non-null  object 
 1   hr            112475 non-null  float64
 2   casual        112475 non-null  int64  
 3   registered    112475 non-null  int64  
 4   temp_c        112475 non-null  float64
 5   feels_like_c  112475 non-null  float64
 6   hum           112475 non-null  float64
 7   windspeed     112475 non-null  float64
 8   weathersit    112475 non-null  int64  
 9   season        112475 non-null  int64  
 10  holiday       112475 non-null  int64  
 11  workingday    112475 non-null  int64  
dtypes: float64(5), int64(6), object(1)
memory usage: 10.3+ MB


In [48]:
def transformData(df):
  df["month"] = df["dteday"].str.split("/").str[0]
  df["year"] = df["dteday"].str.split("/").str[2]
  df["total"] = df["casual"] + df["registered"]

In [49]:
transformData(bikes)

In [50]:
target = ["total"]
X = bikes.drop(columns=["casual", "registered", "dteday", "total"])
y = bikes[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)
norm = MinMaxScaler().fit(X_train)
X_train = norm.transform(X_train)
X_test = norm.transform(X_test)

In [51]:
model = Sequential()
model.add(Dense(64, input_dim=len(X_train[0]), activation='relu'))
model.add(Dense(96, activation='relu'))
model.add(Dropout(0.1))
model.add(Dense(144, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(216, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(144, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(96, activation='relu'))
model.add(Dropout(0.1))
model.add(Dense(64, activation='relu'))

model.add(Dense(1, activation='relu'))

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_48 (Dense)                │ (None, 64)             │           768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_49 (Dense)                │ (None, 96)             │         6,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_30 (Dropout)            │ (None, 96)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_50 (Dense)                │ (None, 144)            │        13,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_31 (Dropout)            │ (None, 144)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_51 (Dense)                │ (None, 216)            │        31,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_32 (Dropout)            │ (None, 216)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_52 (Dense)                │ (None, 144)            │        31,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_33 (Dropout)            │ (None, 144)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_53 (Dense)                │ (None, 96)             │        13,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_34 (Dropout)            │ (None, 96)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_54 (Dense)                │ (None, 64)             │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_55 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 103,737 (405.22 KB)

 Trainable params: 103,737 (405.22 KB)

 Non-trainable params: 0 (0.00 B)

In [52]:
opt = keras.optimizers.Adam(learning_rate=0.0001)
model.compile(loss="mean_squared_error", optimizer=opt, metrics=["mse"])
early_stop = keras.callbacks.EarlyStopping(monitor='val_mse', patience=30)
history = model.fit(X_train, y_train, epochs=200, validation_split=.35, batch_size=20, callbacks=[early_stop],shuffle=False)
hist = pd.DataFrame(history.history)

Epoch 1/200
2559/2559 ━━━━━━━━━━━━━━━━━━━━ 17s 5ms/step - loss: 118270.6797 - mse: 118270.6797 - val_loss: 60034.6133 - val_mse: 60034.6133
Epoch 2/200
2559/2559 ━━━━━━━━━━━━━━━━━━━━ 14s 5ms/step - loss: 59317.0547 - mse: 59317.0547 - val_loss: 48948.4414 - val_mse: 48948.4414
Epoch 3/200
2559/2559 ━━━━━━━━━━━━━━━━━━━━ 14s 5ms/step - loss: 49314.3633 - mse: 49314.3633 - val_loss: 41145.9648 - val_mse: 41145.9648
Epoch 4/200
2559/2559 ━━━━━━━━━━━━━━━━━━━━ 14s 5ms/step - loss: 43126.5547 - mse: 43126.5547 - val_loss: 35398.8047 - val_mse: 35398.8047
Epoch 5/200
2559/2559 ━━━━━━━━━━━━━━━━━━━━ 15s 6ms/step - loss: 38135.4023 - mse: 38135.4023 - val_loss: 30847.7559 - val_mse: 30847.7559
Epoch 6/200
2559/2559 ━━━━━━━━━━━━━━━━━━━━ 14s 5ms/step - loss: 32865.4219 - mse: 32865.4219 - val_loss: 25025.6992 - val_mse: 25025.6992
Epoch 7/200
2559/2559 ━━━━━━━━━━━━━━━━━━━━ 15s 6ms/step - loss: 29047.7988 - mse: 29047.7988 - val_loss: 23483.2754 - val_mse: 23483.2754
Epoch 8/200
2559/2559 ━━━━━━━━━━

In [53]:
predictions = np.round(model.predict(X_test), 2)

rmse = root_mean_squared_error(y_test, predictions)
mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print(rmse, mae, r2)

1055/1055 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
92.4957046508789 57.63087844848633 0.9269176125526428
